# Stage 1 — Pre-Choice EEG Window Feasibility Check

**Goal:** before building any decoder, check (a) how much room we actually have between
stimulus onset and response on each trial, (b) how many trials survive at different
"buffer before response" sizes, and (c) how balanced the choice / accuracy / confidence
classes are — including the 4-way confidence×accuracy split (the one likely to be
data-starved in the incorrect-high-confidence cell).

**You only need `EEG_events_sub-01_run-01.mat` for this notebook** — no raw EEG required yet.

Run every cell top to bottom, then copy/paste the printed text output back into the chat.

In [ ]:
# 1. Upload / locate the events file
from google.colab import files
import os

EVENTS_PATH = "EEG_events_sub-01_run-01.mat"

if not os.path.exists(EVENTS_PATH):
    print("Please upload EEG_events_sub-01_run-01.mat")
    uploaded = files.upload()
    EVENTS_PATH = list(uploaded.keys())[0]

print("Using events file:", EVENTS_PATH)

In [ ]:
# 2. Load and extract event variables
from scipy.io import loadmat
import numpy as np

events = loadmat(EVENTS_PATH)
print("Keys in events file:")
print([k for k in events.keys() if not k.startswith("__")])

confidence = events["confidence"].squeeze()
accuracy   = events["accuracy"].squeeze()
choice     = events["choice"].squeeze()
dotdirection = events["dotdirection"].squeeze()
tstim = events["tstim"].squeeze()
tresp = events["tresp"].squeeze()
tconf = events["tconf"].squeeze()
RT_reported = events["RT"].squeeze()

n_trials = len(confidence)
print(f"\nTotal valid trials: {n_trials}")
print("Unique confidence values:", np.unique(confidence))
print("Unique accuracy values:", np.unique(accuracy))
print("Unique choice values:", np.unique(choice))
print("Unique dotdirection values:", np.unique(dotdirection))

In [ ]:
# 3. Recompute RT from timestamps and sanity-check against the stored RT field
RT_computed = tresp - tstim

diff = RT_computed - RT_reported
print("Max abs difference between computed RT (tresp-tstim) and stored RT field:", np.max(np.abs(diff)))
print("Number of trials where they differ by >5 ms:", np.sum(np.abs(diff) > 5))

print("\nRT (stored field) summary, ms:")
print(f"  min={RT_reported.min():.0f}  max={RT_reported.max():.0f}  mean={RT_reported.mean():.1f}  "
      f"median={np.median(RT_reported):.1f}  sd={RT_reported.std():.1f}")

for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"  {p:>2}th percentile: {np.percentile(RT_reported, p):.0f} ms")

In [ ]:
# 4. Histogram of RT distribution (text summary printed above already covers the numbers;
#    this plot is just a visual sanity check, screenshot it if useful)
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.hist(RT_reported, bins=30, edgecolor="black")
plt.xlabel("Response Time (ms)")
plt.ylabel("Number of trials")
plt.title("RT distribution - Subject 01, Run 01")
plt.grid(True)
plt.show()

In [ ]:
# 5. Window feasibility sweep
# For a pre-response window of fixed LENGTH ending BUFFER ms before tresp,
# check how many trials have a valid window that:
#   - starts no earlier than (tstim - PRE_STIM_BASELINE)   [can't go before recording epoch start]
#   - ends at least BUFFER ms before tresp                  [avoid motor contamination]
#   - has positive length

PRE_STIM_BASELINE = 200   # ms before stimulus onset available as baseline, matches your existing epoch code
WINDOW_LENGTHS = [200, 400, 600, 800]   # candidate window lengths in ms
BUFFERS = [0, 50,100,150,200,250, 300, 350]  # candidate buffers before response, in ms

print(f"{'window_len':>10} {'buffer':>7} {'n_valid':>8} {'pct_valid':>10}")
results = []
for wlen in WINDOW_LENGTHS:
    for buf in BUFFERS:
        window_end = tresp - buf
        window_start = window_end - wlen
        valid = (window_start >= (tstim - PRE_STIM_BASELINE)) & (window_end > window_start)
        n_valid = int(np.sum(valid))
        pct = 100 * n_valid / n_trials
        results.append((wlen, buf, n_valid, pct))
        print(f"{wlen:>10} {buf:>7} {n_valid:>8} {pct:>9.1f}%")

In [ ]:
# 6. Recommended window: show trial-by-trial detail for one specific (window_len, buffer) choice
#    Edit these two values after looking at the table above, then re-run this cell.
CHOSEN_WINDOW_LEN = 600   # ms
CHOSEN_BUFFER = 150       # ms

window_end = tresp - CHOSEN_BUFFER
window_start = window_end - CHOSEN_WINDOW_LEN
valid = (window_start >= (tstim - PRE_STIM_BASELINE)) & (window_end > window_start)

print(f"Chosen window: {CHOSEN_WINDOW_LEN} ms long, ending {CHOSEN_BUFFER} ms before response")
print(f"Valid trials: {int(valid.sum())} / {n_trials} ({100*valid.sum()/n_trials:.1f}%)")
print(f"Trials excluded: {int((~valid).sum())}")

if (~valid).sum() > 0:
    print("\nRT of excluded trials (ms):")
    print(np.sort(RT_reported[~valid]))

In [ ]:
# 7. Class balance checks

print("=== Choice balance ===")
vals, counts = np.unique(choice, return_counts=True)
for v, c in zip(vals, counts):
    print(f"  choice={v}: {c} trials ({100*c/n_trials:.1f}%)")

print("\n=== Accuracy balance ===")
vals, counts = np.unique(accuracy, return_counts=True)
for v, c in zip(vals, counts):
    print(f"  accuracy={v}: {c} trials ({100*c/n_trials:.1f}%)")

print("\n=== Confidence: High (top 3 ratings) vs Low (bottom 3 ratings) vs Medium ===")
# Matches the Gherman & Philiastides (2018) binning approach: 3 highest / 3 lowest of the 9-point scale
sorted_conf = np.sort(np.unique(confidence))
low_labels  = sorted_conf[:3]
high_labels = sorted_conf[-3:]
print("Low confidence ratings treated as:", low_labels)
print("High confidence ratings treated as:", high_labels)

is_low  = np.isin(confidence, low_labels)
is_high = np.isin(confidence, high_labels)
is_med  = ~is_low & ~is_high

print(f"  Low:    {is_low.sum()} trials")
print(f"  Medium: {is_med.sum()} trials")
print(f"  High:   {is_high.sum()} trials")

In [ ]:
# 8. THE key table: 4-way accuracy x confidence split (correct/incorrect x high/low)
# This is the cell most likely to be data-starved — especially incorrect+high.

print(f"{'':>20} {'Low conf':>10} {'High conf':>10}   (Medium-confidence trials excluded from this table)")

for acc_val, acc_label in [(1, 'Correct'), (0, 'Incorrect')]:
    acc_mask = accuracy == acc_val
    n_low  = int(np.sum(acc_mask & is_low))
    n_high = int(np.sum(acc_mask & is_high))
    print(f"{acc_label:>20} {n_low:>10} {n_high:>10}")

n_incorrect_high = int(np.sum((accuracy == 0) & is_high))
print(f"\n>>> incorrect + high-confidence trials (the 'overconfidence' cell): {n_incorrect_high}")
if n_incorrect_high < 15:
    print(">>> WARNING: this cell has very few trials for single-subject decoding.")
    print(">>> Consider: (a) pooling across subjects, (b) using confidence as a continuous")
    print(">>> regressor instead of a 4-way classifier, or (c) a 2-stage design (accuracy first,")
    print(">>> then confidence within each accuracy group).")

## What to send back

Please paste back the full **printed text output** of cells 2, 3, 5, 6, 7, and 8
(no need to paste the histogram image, though a screenshot is welcome too).
That's everything needed to lock in a prediction window and label scheme before we
touch the raw EEG file.